In [0]:
# Widgets
dbutils.widgets.removeAll()

dbutils.widgets.dropdown("env", "dev", ["dev", "prod"], "Ambiente")
dbutils.widgets.text("process_date", "01012026", "Fecha Proceso (DDMMYYYY)")

In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable


env = dbutils.widgets.get("env")
process_date = dbutils.widgets.get("process_date")
catalog = f"{env}_fraud"
gold_path = f"{catalog}.gold.fact_transacciones"
silver_path = f"{catalog}.silver.transacciones_silver"

print(f"🚀 Poblando Fact Table en {gold_path} para el lote {process_date}...")

# 2. Lectura Incremental de Silver
# Leemos solo lo que corresponde a la fecha de proceso actual
df_trans_silver = spark.table(silver_path).filter(col("process_date") == process_date)

# 3. Transformación: Generación de Llave Única (SK) y Selección
# Creamos 'id_transaccion_sk' para que el MERGE sepa exactamente qué registro actualizar o insertar
df_fact_load = df_trans_silver.select(
    sha2(concat(col("id_cliente"), col("fecha_completa"), col("monto")), 256).alias("id_transaccion_sk"),
    "fecha_completa",
    "id_cliente",
    "id_tarjeta",
    col("categoria_mcc").cast("int").alias("codigo_mcc"),
    "monto",
    col("es_fraude").cast("int"),
    "hora_pico",
    "dia_semana",
    "process_date"
)

# 4. DML: Upsert (MERGE) en la tabla física
# Esto evita duplicados si el Job se corre dos veces para la misma fecha
dt_fact = DeltaTable.forName(spark, gold_path)

dt_fact.alias("t").merge(
    df_fact_load.alias("s"),
    "t.id_transaccion_sk = s.id_transaccion_sk"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

# --- BLOQUE DE AUDITORÍA Y RESUMEN ---

# Consultamos la tabla física final
df_final_gold = spark.table(gold_path)
total_gold = df_final_gold.count()
registros_hoy = df_final_gold.filter(col("process_date") == process_date).count()

print("═" * 70)
print(f" 📈 REPORTE DE CARGA GOLD: {gold_path.upper()} ".center(70, "═"))
print("═" * 70)
print(f"{'Total registros acumulados':<35} : {total_gold:,}")
print(f"{'Registros procesados hoy':<35} : {registros_hoy:,}")
print(f"{'Estatus':<35} : ✅ EXITOSO")
print("═" * 70)

# Muestra de los 3 registros más recientes
print(f"\n🔍 VISTA PREVIA DE LA FACT TABLE (Top 3):")
display(df_final_gold.filter(col("process_date") == process_date).limit(3))